In [ ]:
import os
import sys

WORKDIR = os.environ["CONTAINER_WORK_DIR"]

os.chdir(WORKDIR)
sys.path.append(f"{WORKDIR}/test/modules/simul_whisper")
sys.path.append(f"{WORKDIR}/modules/python-utils")
sys.path.append(f"{WORKDIR}/modules/ai-utils")

In [ ]:
from pathlib import Path

In [ ]:
from sj_utils.file.json import load_json

In [ ]:
ESIC_OUTPUT = f"{WORKDIR}/test/performance_test/esic/output/20250730/step1_16b-RTX3090"
LIBRI_OUTPUT = f"{WORKDIR}/test/performance_test/libri/output/dev/20250730/step1_16b-RTX3090"

In [ ]:
esic_output = Path(ESIC_OUTPUT)
libri_output = Path(LIBRI_OUTPUT)
esic_output.exists() and libri_output.exists()

In [ ]:
esic_child = [f.name for f in esic_output.glob("*.json") if f.is_file()]
libri_child = [f.name for f in libri_output.glob("*.json") if f.is_file()]

intersection = set(esic_child) & set(libri_child)
esic_orphan = set(esic_child) - set(libri_child)
libri_orphan = set(libri_child) - set(esic_child)
print("esic_orphan:", len(esic_orphan))
print("libri_orphan:", len(libri_orphan))

In [ ]:
result = {}
for f in intersection:
    _, esic_json = load_json(esic_output / f)
    _, libri_json = load_json(libri_output / f)

    esic_json = esic_json["rt_whisper"]
    libri_json = libri_json["rt_whisper"]

    result[f] = {
        "esic_wer": esic_json["wer_percent"],
        "libri_wer": libri_json["wer_percent"],
        "sum": esic_json["wer_percent"] + libri_json["wer_percent"],
    }

In [ ]:
result = sorted(result.items(), key=lambda x: x[1]["sum"])
for f, data in result:
    print(f"{f}: esic_wer={data['esic_wer']:.2f}, libri_wer={data['libri_wer']:.2f}, sum={data['sum']:.2f}")